Differential privacy mechanisms require bounded sensitivity in order to control the scale of the injected noise [Dwork and Roth, 2014].

In continuous wearable sensor data, such as accelerometer and gyroscope signals, rare extreme peaks and high inter-subject variability are common, particularly during fall events [Sucerquia et al., 2017].

Without bounding the signal range, these extreme values would dominate the sensitivity and lead to excessive noise injection, severely degrading data utility.

Therefore, following standard differential privacy practice, a robust clipping strategy was applied prior to noise injection to bound the signal values and ensure a stable privacy–utility trade-off.

The clipping thresholds were estimated using a high percentile of the signal distribution, in order to reduce the influence of rare extreme outliers while preserving most of the signal dynamics

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.aih_privacy.config import DATA_PROCESSED_DIR
from src.aih_privacy.privacy.dp_features import compute_feature_clips, dp_laplace_on_features_df

In [ ]:
# escolhe o tag que estás a usar (tem de bater com o 03_windowing_features)
tag = "win1.00_step0.50_f5.0"

df_trials_path = DATA_PROCESSED_DIR / "sisfall" / f"df_trials_{tag}.parquet"
subjects_path  = DATA_PROCESSED_DIR / "sisfall" / "subjects_identity.parquet"

df_trials = pd.read_parquet(df_trials_path)
subjects = pd.read_parquet(subjects_path)

df_trials = df_trials.merge(subjects[["subject_id", "pid"]], on="subject_id", how="left", validate="many_to_one")
assert df_trials["pid"].notna().all()

df_trials.head()

,trial_id,label,subject_id,age_group,activity_code,n_windows,c2_max_max,c2_max_mean,c2_max_std,c8_mean,c8_max,c9_mean,c1_max_max,pid
0,D01_SA01_R01,0,SA01,SA,D01,198,0.563236,0.418996,0.037546,0.185930,0.231747,0.237608,1.513698,8b39978f3bf493e0
1,D01_SA02_R01,0,SA02,SA,D01,198,0.471547,0.360264,0.040570,0.188866,0.236531,0.261376,1.535287,8cca10112272b7d0
2,D01_SA03_R01,0,SA03,SA,D01,198,0.753453,0.671717,0.062895,0.190047,0.241245,0.243012,1.474569,5a58367b9ab21a3f
3,D01_SA04_R01,0,SA04,SA,D01,198,0.751347,0.657644,0.056495,0.170927,0.233457,0.218769,1.493332,ac62c148b01bdf2b
4,D01_SA05_R01,0,SA05,SA,D01,198,0.445957,0.358605,0.042250,0.179623,0.236440,0.223999,1.454516,144051cc52779481


In [ ]:
FEATURE_COLS = [
    "c2_max_max",
    "c2_max_mean",
    "c2_max_std",
    "c8_mean",
    "c8_max",
    "c9_mean",
    "c1_max_max",
    "n_windows",
]

LABEL_COL = "label"
GROUP_COL = "pid"  # split por sujeito pseudonimizado

In [4]:
clip_min, clip_max = compute_feature_clips(df_trials, FEATURE_COLS)
clip_min, clip_max

({'c2_max_max': 0.30088988673441636,
  'c2_max_mean': 0.16659257570224956,
  'c2_max_std': 0.03783343274321948,
  'c8_mean': 0.02691918498384299,
  'c8_max': 0.09328334271610883,
  'c9_mean': 0.04210854145834475,
  'c1_max_max': 1.071129933273414,
  'n_windows': 22.0},
 {'c2_max_max': 4.976235242540952,
  'c2_max_mean': 1.289895836094082,
  'c2_max_std': 1.1241133529107603,
  'c8_mean': 0.3574793186503829,
  'c8_max': 1.3369538697627723,
  'c9_mean': 0.9631072637688972,
  'c1_max_max': 5.361700098881207,
  'n_windows': 199.0})

In [ ]:
EPSILONS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
RANDOM_SEED = 42

out_dir = DATA_PROCESSED_DIR / "sisfall" / "privacy"
out_dir.mkdir(parents=True, exist_ok=True)

base_keep = [GROUP_COL, "subject_id", "age_group", "trial_id", "activity_code", LABEL_COL]  # meta

for eps in EPSILONS:
    rng = np.random.default_rng(RANDOM_SEED)

    df_dp = df_trials.copy()
    df_dp[FEATURE_COLS] = dp_laplace_on_features_df(df_trials[FEATURE_COLS], eps, clip_min, clip_max, rng)

    out_path = out_dir / f"df_trials_{tag}_laplace_eps{eps}.parquet"
    df_dp[base_keep + FEATURE_COLS].to_parquet(out_path, index=False)
    print("Saved:", out_path)

Saved: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\dp_trials\df_trials_win1.00_step0.50_f5.0_laplace_eps0.1.parquet
Saved: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\dp_trials\df_trials_win1.00_step0.50_f5.0_laplace_eps0.2.parquet
Saved: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\dp_trials\df_trials_win1.00_step0.50_f5.0_laplace_eps0.5.parquet
Saved: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\dp_trials\df_trials_win1.00_step0.50_f5.0_laplace_eps1.0.parquet
Saved: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\dp_trials\df_trials_win1.00_step0.50_f5.0_laplace_eps2.0.parquet
Saved: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\dp_trials\df_trials_win1.00_step0.50_f5.0_laplace_eps5.0.parquet
Saved: C:\Users\User\Documents\GitHub\aih-privacy\data\processed\sisfall\dp_trials\df_trials_win1.00_step0.50_f5.0_laplace_eps10.0.parquet
